# 1. Stemming



## 1.1 What is stemming ?

Stemming is a text standardization process in Natural Language Processing (NLP) where words are reduced to their base or root form, called the stem.
It works by using simple, rule-based heuristics to chop off suffixes (like "-ing", "-ed", "-s", or "-ly") from the end of a word.

<img src="./images/stemming.png" alt="HTML5 Logo" width="30%">

## 1.2 Why use Stemming?
When building a vocabulary (like we did in the previous step), words like run, running, and runs would normally be treated as completely different tokens with different IDs. Stemming collapses them into a single token (run). This drastically reduces the size of your vocabulary and helps the machine learning model understand that these words represent the same core concept.


### Examples of Stemming

| Original Word | Suffix Removed | Stemmed Token |
| :--- | :--- | :--- |
| `running` | `-ning` / `-ing` | `run` |
| `cats` | `-s` | `cat` |
| `enjoyed` | `-ed` | `enjoy` |
| `happiness` | `-ness` | `happi` (Notice it's not a real word!) |

# 2. Stemming Practical Example

In [4]:
words =  ['eating', 'running', 'eaten', 'writing', 'prograaming', 'programs','history', 'fairly']


### Stemming technique 1: PorterStemmer rule-based algorithm


In [19]:
from nltk.stem import PorterStemmer

In [22]:
stemming = PorterStemmer()
for word in words:
    print(word+ "---->", stemming.stem(word))

eating----> eat
running----> run
eaten----> eaten
writing----> write
prograaming----> prograam
programs----> program
history----> histori
fairly----> fairli


In the above histori is not correct, this is a major disadvantage of stemming. These all will get fixed with Lemmatization (will discuss later).

### Stemming technique 2: RegexStemmer class 


In [7]:
from nltk.stem import RegexpStemmer

In [13]:
reg_stemmer = RegexpStemmer(regexp='ing$|s$|es$|able$', min=4) # Any substring that matches the regular expression will be removed
# last words are ing|s|e|able are removed

print('eating ---->', reg_stemmer.stem('eating'))
print('plans ---->', reg_stemmer.stem('plans'))
print('wishes ---->', reg_stemmer.stem('wishes'))
print('enable ---->', reg_stemmer.stem('enable'))

eating ----> eat
plans ----> plan
wishes ----> wish
enable ----> en


### Stemming technique 3: Snowball Stemmer

The Snowball Stemmer (often called the Porter2 Stemmer) is the direct, upgraded successor to the original Porter Stemmer. It was also created by Martin Porter to fix some of the logical flaws and overly aggressive rules of his original algorithm. nowball supports multiple languages (like Spanish, French, German, and Russian), whereas the original Porter Stemmer was built strictly for English.

In [14]:
from nltk.stem import SnowballStemmer

In [23]:
snowBallStemmer = SnowballStemmer('english')
for word in words:
    print(word+ '----->' + snowBallStemmer.stem(word))

eating----->eat
running----->run
eaten----->eaten
writing----->write
prograaming----->prograam
programs----->program
history----->histori
fairly----->fair


# 3. Lemmatization


**Lemmatization** is a text pre-processing technique in Natural Language Processing (NLP) that reduces words to their base or dictionary form (the **lemma**). 



### How it Works
Unlike stemming, which relies on crude, rule-based chopping (heuristics), lemmatization uses a detailed linguistic dictionary (like WordNet) and **Part-of-Speech (POS) tagging**. By understanding whether a word is being used as a noun, verb, or adjective in a sentence, it can accurately find the correct root word.

### Key Differences: Stemming vs. Lemmatization

| Feature | Stemming | Lemmatization |
| :--- | :--- | :--- |
| **Output** | Often produces non-words (e.g., `studying` -> `studi`). | Always produces a valid dictionary word (e.g., `studying` -> `study`). |
| **Mechanism** | Rule-based suffix stripping. | Dictionary lookups and morphological analysis. |
| **Context** | Ignores context. | Uses context (Part of Speech) to determine the lemma. |
| **Speed** | Very fast and computationally cheap. | Slower and more computationally expensive. |

### Examples of Lemmatization in Action

Notice how lemmatization can completely change the spelling of a word to find its true root, which stemming cannot do:

| Original Word | Context (POS) | Lemma | Notes |
| :--- | :--- | :--- | :--- |
| `better` | Adjective | `good` | Stemming would leave this as `better`. |
| `mice` | Noun | `mouse` | Stemming would just strip the "e" to make `mic`. |
| `meeting` | Verb | `meet` | "I am meeting him." -> `meet` |
| `meeting` | Noun | `meeting` | "The meeting was long." -> `meeting` |

> **Summary:** Use **Stemming** when speed is your top priority (like in massive search engines). Use **Lemmatization** when accuracy and understanding the actual meaning of the text are crucial (like in chatbots, sentiment analysis, or machine translation).

### 3.1 Lemmatization technique - WordNetLemmatizer

In [6]:
from nltk.stem import WordNetLemmatizer
import nltk
nltk.download('wordnet')

[nltk_data] Downloading package wordnet to /Users/hrishi/nltk_data...


True

In [ ]:
lemmaztizer = WordNetLemmatizer()
lemmaztizer.lemmatize('going') # took default pos as noun

'going'

In [12]:
'''
POS- Noun - n
vert -b
adjective - a
adverb - r
'''
lemmaztizer.lemmatize('going', 'v')

'go'

## How to Determine the POS Tag for Lemmatization

To lemmatize accurately, you must provide the context of the word using a Part-of-Speech (POS) tag. You do not do this manually; instead, you use a pre-trained **POS Tagger** to label the words for you.



### Step 1: Automated POS Tagging
First, you pass your tokenized sentence through a tagger (like `nltk.pos_tag`). The tagger looks at the surrounding words and assigns a label to each token. 

For example, the sentence `"The leaves are falling"` might be tagged as:
`[('The', 'DT'), ('leaves', 'NNS'), ('are', 'VBP'), ('falling', 'VBG')]`

### Step 2: The Tag Mapping Problem
There is a catch. Standard POS taggers (like NLTK's default) use the **Penn Treebank** tagset, which has dozens of highly specific tags (e.g., `VBG` for Verb, gerund; `NNS` for Noun, plural). 

However, standard lemmatizers (like the **WordNet Lemmatizer**) only understand four simple, broad categories:
* `n` = Noun
* `v` = Verb
* `a` = Adjective
* `r` = Adverb

### Step 3: Mapping the Tags
To make the tagger and the lemmatizer talk to each other, you have to write a simple mapping function in your code that translates the complex Treebank tags into the simple WordNet tags based on their first letter.

| Treebank Tag (From Tagger) | Starts With | WordNet Tag (For Lemmatizer) | Example |
| :--- | :--- | :--- | :--- |
| `NN`, `NNS`, `NNP`, `NNPS` | `N` | `n` (Noun) | `leaves` -> `leaf` |
| `VB`, `VBD`, `VBG`, `VBN` | `V` | `v` (Verb) | `falling` -> `fall` |
| `JJ`, `JJR`, `JJS` | `J` | `a` (Adjective) | `happiest` -> `happy` |
| `RB`, `RBR`, `RBS` | `R` | `r` (Adverb) | `quickly` -> `quickly` |

> **Pro Tip:** If a word gets tagged as something else (like a preposition or conjunction), or if you don't provide a tag at all, the WordNet lemmatizer will default to treating the word as a **Noun (`n`)**.

### The Modern Alternative: SpaCy
If this two-step mapping process sounds tedious, it is! Modern NLP libraries like **SpaCy** handle POS tagging, mapping, and lemmatization entirely under the hood in a single step. You just feed it a sentence, and it spits out the lemmas automatically.